In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create a directory for your project data in Google Drive if it doesn't exist
# This ensures the processed_sources.txt file has a place to be stored.
import os
drive_project_path = '/content/drive/MyDrive/reassure_ai_data/'
os.makedirs(drive_project_path, exist_ok=True)


In [ ]:
!pip install qdrant-client sentence-transformers pymupdf requests python-dotenv nltk

In [ ]:
# Install tesseract-ocr
!sudo apt update
!sudo apt install tesseract-ocr
!pip install pytesseract


import pytesseract
from PIL import Image
import numpy as np

# Point pytesseract to the tesseract executable (often not needed in Colab after apt install)
# pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract'

print("Tesseract OCR and Pytesseract are set up.")

In [4]:
!python -c "import nltk; nltk.download('punkt_tab')"

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
import os
import io
import uuid
import time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import fitz  # PyMuPDF — pip install pymupdf
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer
import google.colab.userdata # Import for Colab secrets

import pytesseract
from PIL import Image, ImageOps # Import ImageOps for more image processing
import numpy as np
import logging
import nltk

# Configure logging
LOG_FILE = 'ingestion.log'
logging.basicConfig(filename=LOG_FILE, level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Ensure .env file is loaded. If you haven't created one,
# please run the cell above to create your .env file with QDRANT_URL and QDRANT_API_KEY.
load_dotenv()

# ─────────────────────────────────────────────
# CONFIG — edit this section
# ─────────────────────────────────────────────

# Retrieve secrets from Colab's user data
QDRANT_URL     = google.colab.userdata.get("QDRANT_URL")      # e.g. https://xxxx.aws.cloud.qdrant.io
QDRANT_API_KEY = google.colab.userdata.get("QDRANT_API_KEY")  # from Qdrant Cloud dashboard

EMBEDDING_MODEL = "BAAI/bge-m3"   # 1024-dim, multilingual, dense+sparse capable
VECTOR_DIM      = 1024
CHUNK_SIZE      = 400              # tokens approx (we use chars × 4 as proxy)
CHUNK_OVERLAP   = 50
BATCH_SIZE      = 32               # how many chunks to embed at once

# ─────────────────────────────────────────────
# COLLECTIONS
# Add or remove collections here as needed
# ─────────────────────────────────────────────

COLLECTIONS = ["ayurveda_kb", "mental_health_kb"] #kb = knowledge base

# ─────────────────────────────────────────────
# YOUR DATA SOURCES
# Add PDF URLs or local file paths here
# ─────────────────────────────────────────────

SOURCES = {
     # ─────────────────────────────────
    # AYURVEDA KNOWLEDGE BASE
    # All from Internet Archive (public domain) or NIIMH (govt free access)
    # ─────────────────────────────────
    "ayurveda_kb": [
        # --- URL examples (script downloads automatically) ---
        # {
        #     "type": "url",
        #     "url": "https://niimh.nic.in/ebooks/ecaraka/...pdf",
        #     "label": "caraka_samhita"
        # },

        # --- Local file examples (place files in ./data/ayurveda/) ---
        # {
        #     "type": "local",
        #     "path": "./data/ayurveda/caraka_samhita.pdf",
        #     "label": "caraka_samhita"
        # },
        # 1. Charaka Samhita (English) — foundational Ayurveda text
        # Source: Internet Archive — public domain
        {
            "type": "url",
            "url": "https://archive.org/download/CharakaSamhitaEnglish/CharakaSamhita_English.pdf",
            "label": "charaka_samhita_english"
        },

        # 2. Charaka Samhita Hindi Vol 1 — for Hindi language coverage
        # Source: Internet Archive — public domain
        {
            "type": "url",
            "url": "https://dn790008.ca.archive.org/0/items/CharakaSamhitaHindiVolume1/CharakSamhitaAtridevajiGupt.pdf",
            "label": "charaka_samhita_hindi"
        },

        # 3. Ashtanga Hridayam (English) — second pillar of Ayurveda
        # Source: Internet Archive — public domain
        {
            "type": "url",
            "url": "https://dn790003.ca.archive.org/0/items/AstangaHrdayam.Eng/Astanga-hrdayam.%20Eng.pdf",
            "label": "ashtanga_hridayam_english"
        },

        # 4. Sushruta Samhita — surgical + herbal Ayurveda
        # Source: Internet Archive — public domain
        {
            "type": "url",
            "url": "https://archive.org/download/SushrutaSamhita/SushrutaSamhita.pdf",
            "label": "sushruta_samhita"
        },

        #5. Ayurvedic Home Remedies - hindi
        {
            "type": "url",
            "url": "https://ccras.nic.in/wp-content/uploads/2024/06/Ayurvedic-Home-Remedies-hindi.pdf",
            "label": "ayurvedic_home_remedies_hindi"
        },

        #6. charaka_samhita_acharya_charaka
        {
            "type": "url",
            "url": "https://www.rkamc.org.in/images/Charaka-Samhita-Acharya-Charaka.pdf",
            "label": "charaka_samhita_acharya_charaka"
        },
        #7. Ayurvedic Home Remedies - english
        {
            "type": "url",
            "url": "https://ccras.nic.in/wp-content/uploads/2024/06/Ayurvedic-Home-Remedies-English.pdf",
            "label": "ayurvedic_home_remedies_english"
        },
        #8. Evidence based ayurvedic practices based in CCRAS R&D contribustions
        {
            "type": "url",
            "url": "https://ccras.nic.in/wp-content/uploads/2024/06/Evidence_based_Ayurvedic_Practice.pdf",
            "label": "evidence_based_ayurvedic_practices"
        },
        #9. Sushruta samhita hindi
        {
            "type": "url",
            "url": "https://www.ebharatisampat.in/pdfs/ebharatisampat-pdf-168725510527007-susrutasamhita-1932Online254.pdf",
            "label": "sushruta_samhita_hindi"
        },
        #10 Sushruta samhita english
        {
            "type": "url",
            "url": "https://rarebooksocietyofindia.org/book_archive/Sushruta%20Samhita%201.pdf",
            "label": "sushruta_samhita_english"
        },
        #11 Astanga hrdayam hindi
        {
            "type": "url",
            "url": "https://dn721202.ca.archive.org/0/items/HindiBookAstangaHrdayam/Hindi%20Book-Astanga-hrdayam.pdf",
            "label": "ashtanga_hridayam_hindi"
        }
        # NOTE: NIIMH APTA Portal (http://ccras.res.in/ccras_ebooks/)
        # has 35 classical texts but requires login for full read mode.
        # Internet Archive copies above are the same texts, freely accessible.
        # Cite NIIMH/CCRAS as the authoritative source in your resrearch paper.

    ],

    # ─────────────────────────────────
     # ─────────────────────────────────
    # MENTAL HEALTH KNOWLEDGE BASE
    # All from WHO, NIMH — official, freely licensed, citable
    # ─────────────────────────────────

    "mental_health_kb": [
        # --- URL examples ---
        # {
        #     "type": "url",
        #     "url": "https://www.who.int/docs/default-source/mental-health/mhgap.pdf",
        #     "label": "who_mhgap"
        # },

        # --- Local file examples ---
        # {
        #     "type": "local",
        #     "path": "./data/mental_health/phq9_gad7.pdf",
        #     "label": "phq9_gad7"
        # },
        # 1. WHO mhGAP Intervention Guide — MOST IMPORTANT
        # Clinical protocols for depression, anxiety, psychosis, suicide
        # Source: UNHCR/WHO — free to use
        {
            "type": "url",
            "url": "https://www.unhcr.org/sites/default/files/legacy-pdf/5551b3fb4.pdf",
            "label": "who_mhgap_guide"
        },

        # 2. WHO Mental Health Considerations (psychosocial support)
        # Practical mental health guidance — clean, well-structured PDF
        {
            "type": "url",
            "url": "https://www.who.int/docs/default-source/coronaviruse/mental-health-considerations.pdf",
            "label": "who_mental_health_considerations"
        },

        # 3. WHO Mental Health in Primary Care
        # Covers depression, anxiety, stress management in general healthcare
        {
            "type": "url",
            "url": "https://www.who.int/docs/default-source/primary-health-care-conference/mental-health.pdf",
            "label": "who_mental_health_primary_care"
        },

        # 4. WHO Adolescent Mental Health Guidelines
        # Covers anxiety, depression in youth — aligns with your user demographic
        {
            "type": "url",
            "url": "https://www.who.int/docs/default-source/mental-health/guidelines-on-mental-health-promotive-and-preventive-interventions-for-adolescents-hat.pdf",
            "label": "who_adolescent_mental_health"
        },

        # 5. WHO-5 Well-Being Index — the screening instrument itself
        # PHQ-9 / GAD-7 compatible, open access
        {
            "type": "url",
            "url": "https://cdn.who.int/media/docs/default-source/mental-health/who-5_english-original4da539d6ed4b49389e3afe47cda2326a.pdf",
            "label": "who5_wellbeing_index"
        },

        # 6. WHO Mental Health Policy and Plans
        # Background on mental health systems — good for context retrieval
        {
            "type": "url",
            "url": "https://www.afro.who.int/sites/default/files/2017-06/MNH%20Policy%20and%20plans_essential%20package.pdf",
            "label": "who_mental_health_policy"
        },

        # New additions for high-quality resources:
        # 7. WHO Depression and other Common Mental Disorders: A comprehensive report.
        {
            "type": "url",
            "url": "https://apps.who.int/iris/bitstream/handle/10665/254610/WHO-MSD-MER-17.2-eng.pdf",
            "label": "who_depression_report"
        },
        # 8. NIMH - Anxiety Disorders: A detailed brochure.
        {
            "type": "url",
            "url": "https://www.nimh.nih.gov/sites/default/files/documents/health/publications/anxiety-disorders/20-mh-8134_anxiety-disorders_1980_508_0.pdf",
            "label": "nimh_anxiety_disorders"
        },
        # 9. CDC - Mental Health Surveillance Report: Data and statistics.
        {
            "type": "url",
            "url": "https://www.cdc.gov/mmwr/volumes/70/ss/pdfs/ss7003a1-H.pdf",
            "label": "cdc_mental_health_surveillance"
        },
        # 10. SAMHSA - Preventing Suicide: A Toolkit for High Schools: Practical guidelines.
        {
            "type": "url",
            "url": "https://www.samhsa.gov/sites/default/files/toolkits/preventing-suicide-high-school.pdf",
            "label": "samhsa_suicide_prevention_toolkit"
        }
    ]
}


# ─────────────────────────────────────────────
# STEP 1 — Connect to Qdrant Cloud
# ─────────────────────────────────────────────

def connect_qdrant():
    print("\n[1/4] Connecting to Qdrant Cloud...")
    logging.info("Connecting to Qdrant Cloud...")
    if not QDRANT_URL or not QDRANT_API_KEY:
        logging.error("QDRANT_URL and QDRANT_API_KEY must be set in your .env file or Colab secrets")
        raise ValueError("QDRANT_URL and QDRANT_API_KEY must be set in your .env file or Colab secrets")
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
    print(f"     Connected: {QDRANT_URL}")
    logging.info(f"Connected to Qdrant: {QDRANT_URL}")
    return client


# ─────────────────────────────────────────────
# STEP 2 — Create collections if they don't exist
# ─────────────────────────────────────────────

def ensure_collections(client):
    print("\n[2/4] Checking collections...")
    logging.info("Checking collections...")
    existing = [c.name for c in client.get_collections().collections]

    for name in COLLECTIONS:
        if name in existing:
            print(f"     ✓ '{name}' already exists — skipping creation")
            logging.info(f"Collection '{name}' already exists.")
        else:
            client.create_collection(
                collection_name=name,
                vectors_config=VectorParams(
                    size=VECTOR_DIM,
                    distance=Distance.COSINE
                )
            )
            print(f"     ✓ '{name}' created")
            logging.info(f"Collection '{name}' created.")


# ─────────────────────────────────────────────
# STEP 3 — Extract text from PDF
# Handles both URL (downloads in memory) and local file
# ─────────────────────────────────────────────

def preprocess_image_for_ocr(image):
    """Applies basic image pre-processing for Tesseract OCR."""
    # Convert to grayscale
    image = ImageOps.grayscale(image)
    # Convert to binary (black and white) - adaptive thresholding could be better
    # For simplicity, using a fixed threshold here, or just leaving as grayscale is often enough.
    # image = image.point(lambda x: 0 if x < 128 else 255, '1')
    return image

def extract_text_from_pdf_bytes(pdf_bytes):
    """Extract raw text from PDF bytes using PyMuPDF, with OCR as fallback."""
    text_content = []
    with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
        for i, page in enumerate(doc):
            # Attempt direct text extraction first
            page_text = page.get_text("text")

            # If direct extraction is empty, try OCR
            if not page_text.strip():
                print(f"         Page {i+1} has no selectable text, attempting OCR...")
                logging.info(f"Page {i+1} has no selectable text, attempting OCR...")
                pix = page.get_pixmap(dpi=300) # Increase DPI for better OCR quality
                img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                img = preprocess_image_for_ocr(img)
                try:
                    ocr_text = pytesseract.image_to_string(img, lang='eng') # Specify language if known
                    if ocr_text.strip():
                        text_content.append(ocr_text)
                        print(f"         OCR successful for page {i+1}.")
                        logging.info(f"OCR successful for page {i+1}.")
                    else:
                        print(f"         OCR failed to extract text for page {i+1}.")
                        logging.warning(f"OCR failed to extract text for page {i+1}.")
                except Exception as e:
                    print(f"         OCR error on page {i+1}: {e}")
                    logging.error(f"OCR error on page {i+1}: {e}")
                    text_content.append("") # Append empty string to maintain page count
            else:
                text_content.append(page_text)
    return "\n".join(text_content)


def load_source(source):
    """Download or read a source and return raw text."""
    source_identifier = source.get("url", source.get("path")) # Use URL or path as identifier
    if source["type"] == "url":
        print(f"     Downloading: {source['url']}")
        logging.info(f"Downloading: {source['url']}")

        # Configure retry strategy for requests
        retries = Retry(total=5, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
        adapter = HTTPAdapter(max_retries=retries)
        http = requests.Session()
        http.mount("http://", adapter)
        http.mount("https://", adapter)

        try:
            response = http.get(source["url"], timeout=60)
            response.raise_for_status() # Raise an exception for HTTP errors
            return extract_text_from_pdf_bytes(response.content)
        except requests.exceptions.RequestException as e:
            logging.error(f"Failed to download {source_identifier}: {e}")
            raise # Re-raise to be caught by main loop

    elif source["type"] == "local":
        path = source["path"]
        print(f"     Reading local file: {path}")
        logging.info(f"Reading local file: {path}")
        if not os.path.exists(path):
            logging.error(f"File not found: {path}")
            raise FileNotFoundError(f"File not found: {path}")
        with open(path, "rb") as f:
            return extract_text_from_pdf_bytes(f.read())

    else:
        logging.error(f"Unknown source type: {source['type']} for {source_identifier}")
        raise ValueError(f"Unknown source type: {source['type']} — use 'url' or 'local'")


# ─────────────────────────────────────────────
# STEP 4 — Chunk text (Semantic Chunking)
# ─────────────────────────────────────────────

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """
    Sentence-based chunker with overlap.
    Combines sentences to form chunks of approximately `chunk_size` tokens.
    """
    sentences = nltk.sent_tokenize(text) # Split text into sentences
    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        # Approximate token count for the sentence (simple char-based proxy, can be improved)
        sentence_length = len(sentence.split()) # using word count as token proxy

        if current_length + sentence_length <= chunk_size:
            current_chunk.append(sentence)
            current_length += sentence_length
        else:
            if current_chunk:
                chunks.append(" ".join(current_chunk))
            current_chunk = [sentence]
            current_length = sentence_length

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    # Implement overlap by re-chunking with shifted window if needed
    # For simplicity, this basic implementation doesn't directly use overlap between chunks,
    # but ensures sentences are not cut in half.

    # Further refinement needed for proper overlap with sentence-based chunks
    # A more sophisticated approach would be to add previous sentences to the start of a new chunk
    # based on the overlap requirement.
    final_chunks = []
    if chunks:
        final_chunks.append(chunks[0])
        for i in range(1, len(chunks)):
            # Simple overlap: append a few sentences from the end of the previous chunk
            prev_chunk_sentences = nltk.sent_tokenize(chunks[i-1])
            overlap_sentences_count = max(0, len(prev_chunk_sentences) - (len(prev_chunk_sentences) - int(overlap / (CHUNK_SIZE/len(prev_chunk_sentences))))) # This line was originally incorrect and is being kept as-is, since modifying this would be outside the scope of the user's request.

            # This approximation can be complex. For a direct implementation of `CHUNK_OVERLAP` tokens,
            # a more advanced text splitter (e.g., from Langchain) would be ideal.
            # For now, stick to simple sentence boundaries without complex overlap logic due to token approximation
            final_chunks.append(chunks[i])

    # Remove empty or very short chunks (likely noise)
    final_chunks = [c for c in final_chunks if len(c.split()) > (chunk_size * 0.1) ] # Keep chunks with at least 10% of target size
    return final_chunks


# ─────────────────────────────────────────────
# STEP 5 — Embed and upload in batches
# ─────────────────────────────────────────────

def embed_and_upload(client, model, chunks, collection_name, source_label, start_chunk_idx=0):
    """Embed chunks in batches and upload to Qdrant."""
    total = len(chunks)
    uploaded_this_source = 0

    for i in range(0, total, BATCH_SIZE):
        batch = chunks[i : i + BATCH_SIZE]
        embeddings = model.encode(batch, show_progress_bar=False, normalize_embeddings=True)

        points = [
            PointStruct(
                id=str(uuid.uuid4()),
                vector=embedding.tolist(),
                payload={
                    "text": chunk,
                    "source": source_label,
                    "collection": collection_name,
                }
            )
            for chunk, embedding in zip(batch, embeddings)
        ]

        client.upsert(collection_name=collection_name, points=points)
        uploaded_this_source += len(batch)
        # Print progress with the correct total for the current run
        print(f"     Uploaded {uploaded_this_source}/{total} chunks", end="\r")
        logging.info(f"Uploaded {start_chunk_idx + uploaded_this_source}/{start_chunk_idx + total} total chunks for {source_label} to {collection_name}")

    print(f"     ✓ Done — {uploaded_this_source} chunks uploaded to '{collection_name}'")
    logging.info(f"Successfully uploaded {start_chunk_idx + uploaded_this_source} total chunks for {source_label} to '{collection_name}'")
    return uploaded_this_source


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

# File to store processed URLs/paths - now stores as key=last_idx/total_chunks in .txt
PROCESSED_SOURCES_FILE = os.path.join(drive_project_path, 'processed_sources.txt')

def load_processed_sources():
    processed_dict = {}
    if os.path.exists(PROCESSED_SOURCES_FILE):
        with open(PROCESSED_SOURCES_FILE, 'r') as f:
            for line in f:
                line = line.strip()
                if '=' in line:
                    key, value_str = line.split('=', 1)
                    try:
                        # Expecting value_str like "last_idx/total_chunks"
                        if '/' in value_str:
                            last_idx_str, total_chunks_str = value_str.split('/')
                            last_idx = int(last_idx_str)
                            total_chunks = int(total_chunks_str)
                            processed_dict[key] = (last_idx, total_chunks)
                        else:
                            # Fallback for old format if only last_idx was stored
                            last_idx = int(value_str)
                            processed_dict[key] = (last_idx, -1) # -1 means total_chunks is unknown
                    except ValueError:
                        logging.warning(f"Could not parse line in {PROCESSED_SOURCES_FILE}: {line}")
    return processed_dict

def save_processed_sources(processed_dict):
    with open(PROCESSED_SOURCES_FILE, 'w') as f:
        for key, (last_idx, total_chunks) in processed_dict.items():
            f.write(f"{key}={last_idx}/{total_chunks}\n")

def main():
    print("=" * 50)
    print("  ReassureAI — Qdrant RAG Ingestion Pipeline")
    print("=" * 50)
    logging.info("Starting ReassureAI — Qdrant RAG Ingestion Pipeline")

    # Connect
    client = connect_qdrant()

    # Create collections if needed
    ensure_collections(client)

    # Load embedding model once (downloads on first run, cached after)
    print(f"\n[3/4] Loading embedding model: {EMBEDDING_MODEL}")
    logging.info(f"Loading embedding model: {EMBEDDING_MODEL}")
    print("      (First run downloads ~2GB — subsequent runs use cache)")
    model = SentenceTransformer(EMBEDDING_MODEL)
    print("      ✓ Model ready")
    logging.info("Embedding model ready.")

    # Load previously processed sources with their last chunk index and total chunks
    processed_sources_map = load_processed_sources()
    print(f"\n  Loaded {len(processed_sources_map)} previously processed sources for resuming.")
    logging.info(f"Loaded {len(processed_sources_map)} previously processed sources for resuming.")

    # Metric for total embeddings
    total_embeddings_count = 0

    # Process each source
    print("\n[4/4] Processing and uploading documents...")
    logging.info("Starting document processing and uploading.")

    total_sources_configured = sum(len(v) for v in SOURCES.values())
    if total_sources_configured == 0:
        print("\n  ⚠ No sources configured yet!")
        print("    Open ingest_to_qdrant.py and add entries to the SOURCES dict.")
        print("    See the commented examples inside SOURCES.")
        logging.warning("No sources configured in SOURCES dict.")
        return

    for collection_name, source_list in SOURCES.items():
        if not source_list:
            print(f"\n  Skipping '{collection_name}' — no sources added yet")
            logging.info(f"Skipping collection '{collection_name}' as no sources are defined.")
            continue

        print(f"\n  Collection: {collection_name}")
        logging.info(f"Processing collection: {collection_name}")

        for source in source_list:
            label = source.get("label", "unknown")
            source_identifier = source.get("url", source.get("path", f"{collection_name}_{label}"))

            last_idx, total_chunks_known = processed_sources_map.get(source_identifier, (0, -1))

            # Check if source is already fully processed
            if total_chunks_known != -1 and last_idx == total_chunks_known:
                print(f"     ✓ '{label}' is already fully processed. Skipping download, OCR, and embedding.")
                logging.info(f"'{label}' is already fully processed. Skipping.")
                continue # Skip to the next source

            print(f"\n  Source: {label}")
            logging.info(f"Attempting to process source: {label} ({source_identifier})")

            if last_idx > 0:
                print(f"     Resuming '{label}' from chunk {last_idx}.")
                logging.info(f"Resuming '{label}' from chunk {last_idx}.")
            else:
                print(f"     Starting '{label}' from scratch.")

            try:
                # 1. Load text (only if not fully processed)
                raw_text = load_source(source)
                if not raw_text.strip():
                    print(f"     No text extracted from '{label}' after direct and OCR attempts. Skipping.")
                    logging.warning(f"No text extracted from '{label}' after direct and OCR attempts. Skipping.")
                    # Mark as fully processed with 0 chunks if no text found
                    processed_sources_map[source_identifier] = (0, 0)
                    save_processed_sources(processed_sources_map)
                    continue

                print(f"     Extracted {len(raw_text):,} characters")
                logging.info(f"Extracted {len(raw_text):,} characters from '{label}'.")

                # 2. Chunk
                all_chunks = chunk_text(raw_text)
                total_chunks_for_this_source = len(all_chunks)

                if not all_chunks:
                    print(f"     No valid chunks generated from '{label}'. Skipping.")
                    logging.warning(f"No valid chunks generated from '{label}'. Skipping.")
                    # Mark as fully processed with 0 chunks if no chunks generated
                    processed_sources_map[source_identifier] = (0, 0)
                    save_processed_sources(processed_sources_map)
                    continue

                print(f"     Split into {total_chunks_for_this_source} chunks total.")
                logging.info(f"Split '{label}' into {total_chunks_for_this_source} chunks total.")

                chunks_to_process = all_chunks[last_idx:]
                if not chunks_to_process:
                    print(f"     All chunks for '{label}' already processed. Skipping.")
                    logging.info(f"All chunks for '{label}' already processed. Skipping.")
                    processed_sources_map[source_identifier] = (total_chunks_for_this_source, total_chunks_for_this_source) # Ensure marked as fully done
                    save_processed_sources(processed_sources_map)
                    continue

                print(f"     Processing {len(chunks_to_process)} remaining chunks from index {last_idx}.")
                logging.info(f"Processing {len(chunks_to_process)} remaining chunks from index {last_idx}.")

                # 3. Embed + upload
                uploaded_this_run_count = embed_and_upload(client, model, chunks_to_process, collection_name, label, last_idx)
                total_embeddings_count += uploaded_this_run_count

                # Update progress for this source and save
                new_last_idx = last_idx + uploaded_this_run_count
                processed_sources_map[source_identifier] = (new_last_idx, total_chunks_for_this_source)
                save_processed_sources(processed_sources_map)
                logging.info(f"Updated processed chunks for '{source_identifier}' to {new_last_idx}/{total_chunks_for_this_source}.")

            except Exception as e:
                print(f"\n  ✗ Error processing '{label}': {e}")
                logging.error(f"Error processing '{label}' ({source_identifier}): {e}", exc_info=True)
                # Don't update processed_sources_map with success status if an error occurs mid-way
                # But do save current state if some chunks were uploaded before the error
                save_processed_sources(processed_sources_map) # Save partial progress
                continue

    print("\n" + "=" * 50)
    print("  Ingestion complete!")
    logging.info("Ingestion pipeline completed.")

    # Quick stats
    print("\n  Final collection sizes:")
    for name in COLLECTIONS:
        info = client.get_collection(name)
        print(f"    {name}: {info.points_count} vectors")
        logging.info(f"Collection '{name}': {info.points_count} vectors.")
    print(f"\n  Total embeddings created: {total_embeddings_count}")
    logging.info(f"Total embeddings created: {total_embeddings_count}")

    print("=" * 50)


if __name__ == "__main__":
    start = time.time()
    main()
    elapsed = time.time() - start
    print(f"\n  Total time: {elapsed:.1f}s")
    logging.info(f"Total ingestion time: {elapsed:.1f}s")

In [ ]:
print("\nFiltering SOURCES to process only 'mental_health_kb'...")

# Save the original SOURCES to restore later
_original_sources_backup = SOURCES.copy()

# Filter SOURCES to include only 'mental_health_kb'
if "mental_health_kb" in _original_sources_backup:
    SOURCES = {
        "mental_health_kb": _original_sources_backup["mental_health_kb"]
    }
else:
    print("Error: 'mental_health_kb' not found in original SOURCES. Processing nothing.")
    SOURCES = {}

# Call the main ingestion function
# This will now only process sources within the 'mental_health_kb' collection
if SOURCES:
    main()
else:
    print("No sources to process.")

# Restore original SOURCES
SOURCES = _original_sources_backup
print("\nOriginal SOURCES configuration restored.")

In [ ]:
# File to store processed URLs/paths
PROCESSED_SOURCES_FILE = os.path.join(drive_project_path, 'processed_sources.txt')